# 03 · Could a supervised model take over? (a dead end)

After K-means worked, I asked: if I take the groups that
K-means found in some leaves, can a **k-nearest-neighbours** classifier learn to predict the group of a pixel in a new
leaf, without running K-means again?

Short answer: it reached an accuracy of about **0.80** and I dropped it. A classifier trained on K-means' own labels
can at best imitate K-means, and it adds an extra step. I keep it here because it is part of the path.

Cleaned-up version of the "KNN" section of `ViruDetRGB-a0.2`. Needs the scans in `data/images/` (not included);
saved **without outputs**.

What the original did, and what to keep in mind:

* **19 leaves** were clustered with the standardised 6-feature K-means (3 colours + 3 differences), 3 groups each.
* The labels used for training are K-means' **own raw numbering**, which changes from one image to the next (see
  notebook 02, section 6). So the classifier was trained on labels that did not mean the same thing in every leaf.
  `ORDER_LABELS = False` reproduces that; set it to `True` to use the ordered groups instead (not something I ran).
* 80 % / 20 % train/test split, stratified. The original had no random seed (results changed on every run); I fixed one.
* The classifier was trained on the **unscaled** features although a scaled copy was computed (and never used).
* In the original, results were: accuracy 0.798 with 3 neighbours and 0.795 with 6. The cells of the notebook that
  predicted leaf 139 pass 8 columns to a model trained on 6, so they cannot have run in the state in which they were
  saved, and a third experiment with 7 neighbours refers to a variable (`knn_12`) that is never defined. I did not
  carry them over as they were.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

IMAGES_DIR = Path("../../data/images")              # not included: vir_<id>.jpg
TRAINING_LEAVES = [100, 105, 112, 114, 117, 121, 123, 126, 127, 128, 13, 131, 133, 134, 135, 137, 14, 140, 141]
PROBLEM_LEAF = 139                                   # a leaf where the raw-feature K-means had failed
FEATURES = ["red", "green", "blue", "diff_r_g", "diff_r_b", "diff_b_g"]
ORDER_LABELS = False
SEED = 2206


def leaf_table(leaf_id):
    """Pixels of one leaf with the standardised 6-feature K-means label (3 groups)."""
    img = cv2.cvtColor(cv2.imread(str(IMAGES_DIR / f"vir_{leaf_id}.jpg")), cv2.COLOR_BGR2RGB)
    img = cv2.medianBlur(img, 11).astype(int)
    df = pd.DataFrame({"red": img[..., 0].ravel(), "green": img[..., 1].ravel(), "blue": img[..., 2].ravel()})
    df["diff_r_g"] = df["red"] - df["green"]
    df["diff_r_b"] = df["red"] - df["blue"]
    df["diff_b_g"] = df["blue"] - df["green"]
    scaled = preprocessing.StandardScaler().fit_transform(df[FEATURES])
    km = KMeans(n_clusters=3, random_state=SEED, n_init=10).fit(scaled)
    labels = km.labels_
    if ORDER_LABELS:
        order = np.argsort(km.cluster_centers_.sum(axis=1))
        labels = np.vectorize({old: new for new, old in enumerate(order)}.get)(labels)
    df["kmeans"] = labels
    return df, img.shape[:2]


tables = {leaf_id: leaf_table(leaf_id)[0] for leaf_id in TRAINING_LEAVES}
combined = pd.concat(tables.values(), ignore_index=True)
X, y = combined[FEATURES], combined["kmeans"]
print(combined.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

models = {}
for k in (3, 6):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    models[k] = knn
    pred = knn.predict(X_test)
    print(f"{k} neighbours: accuracy = {accuracy_score(y_test, pred):.3f}")
    ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap=plt.cm.Blues)
    plt.title(f"kNN, {k} neighbours"); plt.show()

## A problematic leaf

Leaf 139 is one where the raw-feature K-means had given a severity near 90 % (see notebook 02, section 8).
I predicted its pixels with the classifier and looked at the picture. I did not pursue this further.

In [ ]:
table, shape = leaf_table(PROBLEM_LEAF)
fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(cv2.cvtColor(cv2.imread(str(IMAGES_DIR / f"vir_{PROBLEM_LEAF}.jpg")), cv2.COLOR_BGR2RGB)); ax[0].set_title("Leaf")
ax[1].imshow(table["kmeans"].to_numpy().reshape(shape)); ax[1].set_title("K-means (standardised)")
ax[2].imshow(models[3].predict(table[FEATURES]).reshape(shape)); ax[2].set_title("kNN, 3 neighbours")
for a in ax: a.axis("off")
plt.show()

Next: [`04_v2_branch_scans.ipynb`](04_v2_branch_scans.ipynb) — from one leaflet to a whole branch (2025).